# Guided Lab：Python 學習助理與 Memory Store

**時間：20 分鐘｜情境：依使用者偏好解說泛型程式設計**

兩位使用者詢問同一個問題：

> Python 的 `TypeVar` 與 `Generic` 有什麼用途？請用泛型類別 `Box[T]` 說明。

| 使用者 | 回覆語言 | 回答風格 |
|---|---|---|
| `user_123` | 台灣繁體中文 | 精簡、直接 |
| `user_456` | English | detailed, step-by-step |

你要把偏好存入 Memory Store，讓 Python 學習助理依目前使用者套用偏好。
泛型程式設計是提問內容；本次練習重點是 Store 與動態提示詞。

完成後，你能：

1. 用 namespace 與 key 儲存、讀取使用者偏好。
2. 用 runtime context 與 Middleware 將偏好帶入 System Prompt。
3. 根據中間資料，解釋不同使用者與偏好更新後的處理流程。

先備能力：已閱讀 Memory Store（一）講義，能閱讀 Python 字典、tuple、函式與 dataclass。

| 時間 | 活動 |
|---|---|
| 0–2 分鐘 | 閱讀情境與資料流 |
| 2–6 分鐘 | 建立與操作 Store |
| 6–12 分鐘 | 串接 Middleware 與 Agent |
| 12–17 分鐘 | 比較兩位使用者，再更新一項偏好 |
| 17–20 分鐘 | 解釋資料流並繳交 |


## 1. 準備與元件責任

請備妥課程 Python kernel、`langchain`、`langgraph`、`langchain-openai`、`python-dotenv` 與可用的模型金鑰。課堂不進行套件安裝。

| 元件 | 本次責任 |
|---|---|
| Static runtime context | 每次呼叫時提供目前使用者的 `user_id` |
| Memory Store | 保存使用者的語言與回答風格 |
| Middleware | 讀取偏好，組合動態 System Prompt |
| Agent State 的 `messages` | 提供本次問題與保存本次回覆 |
| 模型 | 根據提示詞與問題產生回答 |

資料流：**context 的 user_id → namespace → Store 偏好 → Middleware → System Prompt → 模型**。

以下資料由程式預先建立，模擬已保存的使用者偏好。
`prompt_events` 是課堂觀察清單，用來記錄 Middleware 處理的中間資料。

依序執行。`__Fxx__` 是填空標記，先填完該格再執行；未填完會失敗。
若要完整重做，重新啟動 kernel 並從頭執行，恢復初始偏好。


### 環境設定

使用既有環境變數或課程專案的 `.env` 載入金鑰，程式只檢查是否存在。



In [ ]:
# Load API Key
from getpass import getpass
import os
from pathlib import Path

from dotenv import load_dotenv

dotenv_path = Path('your_env_file_path/.env')  # Replace with the path to your .env file
# load_dotenv(dotenv_path=dotenv_path)  # take environment variables from .env.
load_dotenv()  # take environment variables from .env.

# 可於執行時隱藏輸入。
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("請輸入 OpenAI API key：")
# 可換成其他可用模型
model_name = "openai:gpt-5-nano"
print("環境設定已載入。")


### 共用設定（教師提供）

`UserStaticContext` 定義本次呼叫的使用者資訊。
`BASE_SYSTEM_PROMPT` 定義學習助理的教學責任；語言與風格稍後由 Middleware 加入。
`QUESTION` 是三次比較都使用的問題。


In [ ]:
import json
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.messages import SystemMessage
from langgraph.store.memory import InMemoryStore

@dataclass
class UserStaticContext:
    user_id: str

BASE_SYSTEM_PROMPT = """
你是一位 Python 學習助理，協助學生理解程式設計概念。

回答要正確、清楚；不確定時明確說明。
""".strip()

QUESTION = "Python 的 TypeVar 與 Generic 有什麼用途？請用泛型類別 Box[T] 說明。"
KEY = "communication"
prompt_events = []


## 2. 儲存與查看偏好（4 分鐘）

每位使用者有自己的 namespace，兩位使用者都用 `communication` 作為 key。

填入 **F01–F02**：

| 填空 | 提示 |
|---|---|
| F01 | 建立保存在記憶體中的 Store |
| F02 | 使用 Store 的儲存／更新方法 |

輸入是兩位使用者的偏好字典；執行後，Store 各保存一份偏好文件。


In [ ]:
store = __F01__

__F02__(
    namespace=("users", "user_123", "preferences"),
    key=KEY,
    value={
        "preferred_language": "台灣繁體中文",
        "preferred_answer_style": "精簡、直接",
    },
)

store.put(
    namespace=("users", "user_456", "preferences"),
    key=KEY,
    value={
        "preferred_language": "English",
        "preferred_answer_style": "detailed, step-by-step",
    },
)


### 查看 Store（教師提供）

`search()` 依 namespace 前綴查詢，以下最多列出 20 筆。

每個回傳物件的 `namespace`、`key`、`value` 分別顯示資料空間、文件識別碼與內容。

執行後確認：兩位使用者的 key 是否相同？namespace 是否相同？


In [ ]:
items = store.search(("users",), limit=20)
for item in items:
    print("namespace:", item.namespace)
    print("key:", item.key)
    print("value:", json.dumps(item.value, ensure_ascii=False))
    print()


## 3. 串接動態提示詞（6 分鐘）

Middleware 在模型呼叫時取得 `ModelRequest`。

它的 `runtime.context` 提供自訂的 Static Context，`runtime.store` 提供 Agent 配置的 Store。

填入 **F03–F04**：

| 填空 | 提示 |
|---|---|
| F03 | 從 request 的 runtime context 取得 user_id |
| F04 | 從 runtime_store 讀取指定 namespace 與 key 的單筆資料 |

回傳的 `Item.value` 是偏好字典；查無資料時使用預設偏好。

本例基礎提示詞是純文字，使用 `request.system_message.content` 取得，再加入偏好指示。

函式回傳新的 `SystemMessage`，由 `@dynamic_prompt` 交給模型呼叫流程。


In [ ]:
@dynamic_prompt
def user_dynamic_prompt(request: ModelRequest[UserStaticContext]) -> SystemMessage:
    user_id = __F03__
    runtime_store = request.runtime.store
    if runtime_store is None:
        raise RuntimeError("Agent 尚未配置 Store。")

    namespace = ("users", user_id, "preferences")
    comm_pref = __F04__(namespace=namespace, key=KEY)
    preferences = comm_pref.value if comm_pref is not None else {}

    language = preferences.get("preferred_language", "台灣繁體中文")
    style = preferences.get("preferred_answer_style", "精簡、直接")
    style_instruction = f"請使用 {language} 回覆，並且回答的內容要 {style}。"

    base_prompt = (
        request.system_message.content if request.system_message is not None else ""
    )
    dynamic_system_prompt = base_prompt.strip() + "\n" + style_instruction

    # 教師提供的觀察紀錄：保留本次實際讀取的資料與產生的提示詞。
    prompt_events.append({
        "user_id": user_id,
        "namespace": namespace,
        "key": KEY,
        "preferences": dict(preferences),
        "system_prompt": dynamic_system_prompt,
    })
    return SystemMessage(content=dynamic_system_prompt)


### 建立 Agent

填入 **F05–F07**，把已建立的元件交給 Agent。

| 填空 | 提示 |
|---|---|
| F05 | 指定 static runtime context 的 schema |
| F06 | 提供已保存偏好的 Store 物件 |
| F07 | 用 list 提供動態提示詞 Middleware |

本例每次呼叫都傳入相同問題，使用同一個 Store 取得偏好。


In [ ]:
agent = create_agent(
    model=model_name,
    system_prompt=BASE_SYSTEM_PROMPT,
    tools=[],
    context_schema=__F05__,
    store=__F06__,
    middleware=__F07__,
)


### 觀察函式（教師提供）

`show_observation()` 顯示本次新增的 Middleware 紀錄，再顯示最後的模型回答。
`event_start` 是呼叫前的紀錄數量，用來只列出這次呼叫的資料。

偏好與 System Prompt 是核對資料流的主要證據；回答用來觀察模型如何遵循提示。


In [ ]:
def show_observation(result, event_start):
    new_events = prompt_events[event_start:]
    if not new_events:
        raise RuntimeError("本次沒有 Middleware 紀錄，請檢查 Agent 的 middleware 設定。")
    for event in new_events:
        print("使用者:", event["user_id"])
        print("namespace:", event["namespace"])
        print("key:", event["key"])
        print("讀取的偏好:", json.dumps(event["preferences"], ensure_ascii=False))
        print("動態 System Prompt:\n", event["system_prompt"])
    print("\n模型回答:\n", result["messages"][-1].content)


## 4. 預測、執行與比較（5 分鐘）

**先填預測，再執行下一格。**

1. `user_123` 的提示詞會加入哪種語言與風格？ **你的預測：＿＿＿**
2. 改成 `user_456` 後，查詢的 namespace 哪個部分會改變？ **你的預測：＿＿＿**

填入 **F08**：在 `context` 傳入 `user_123` 的 `UserStaticContext` 物件。
`messages` 只傳入相同問題，使用者身分由 context 提供。


In [ ]:
event_start = len(prompt_events)
result_123 = agent.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    context=__F08__,
)
show_observation(result_123, event_start)


### 比較另一位使用者（教師提供呼叫程式）

保持 Agent、問題與 key 相同，改用 `user_456` 的 context。


In [ ]:
event_start = len(prompt_events)
result_456 = agent.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    context=UserStaticContext(user_id="user_456"),
)
show_observation(result_456, event_start)


### 記錄觀察

| 核對項目 | user_123 | user_456 |
|---|---|---|
| 實際查詢的 namespace | ＿＿＿ | ＿＿＿ |
| System Prompt 中的語言與風格 | ＿＿＿ | ＿＿＿ |
| 回答是否符合偏好？指出一個例子 | ＿＿＿ | ＿＿＿ |

### 更新一項偏好

維持 `user_123`、問題、namespace、key 與語言相同，只把回答風格改為「詳細、分步說明」。
使用相同 namespace 與 key 的 `put()` 更新文件時，要一併保留原本的語言欄位。

**執行前預測：**下一次呼叫的 System Prompt 會改變哪一段？為什麼？ **＿＿＿**


In [ ]:
store.put(
    namespace=("users", "user_123", "preferences"),
    key=KEY,
    value={
        "preferred_language": "台灣繁體中文",
        "preferred_answer_style": "詳細、分步說明",
    },
)

event_start = len(prompt_events)
result_updated = agent.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    context=UserStaticContext(user_id="user_123"),
)
show_observation(result_updated, event_start)


**更新後核對：**

- 從偏好紀錄與 System Prompt 指出更新已被讀取的證據：**＿＿＿**
- 與第一次回答比較，指出一個風格差異；若沒有明顯差異，也如實記錄：**＿＿＿**

## 5. 解釋與繳交（3 分鐘）

用 1–2 句回答每題：

1. `user_id` 與使用者偏好分別由哪個元件提供？ **＿＿＿**
2. 更新 Store 後，新的偏好如何進入模型的輸入？請指出觀察到的證據。 **＿＿＿**

繳交完成的 notebook，保留八個填空的程式、預測、三次呼叫的觀察與兩題解釋。
儲存前確認 notebook 沒有金鑰等敏感資訊。


## 參考資料

- 原始案例：[memory_store_case_study_1.ipynb](../../notebooks/memory/memory_store_case_study_1.ipynb)
- [Memory Store（一）講義](/Users/hychen39/0work1/Teaching/ai_agent_dev/lecture_notes/memory/memory_store_1.md)
- [Stores — Docs by LangChain](https://docs.langchain.com/oss/python/langgraph/stores)
- [Runtime — Docs by LangChain](https://docs.langchain.com/oss/python/langchain/runtime)
- [Dynamic prompt — Docs by LangChain](https://docs.langchain.com/oss/python/langchain/middleware/custom#dynamic-prompt)

本 lab 使用目前的 `request.system_message` 取得基礎提示詞。
